In [28]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import joblib

In [29]:
data = pd.read_csv('Fuel_Consumption_2000-2022.csv')
# Rename columns to match the model's expected format
data = data.rename(columns={
    'VEHICLE CLASS': 'Vehicle class',
    'ENGINE SIZE': 'Engine size (L)',
    'CYLINDERS': 'Cylinders',
    'TRANSMISSION': 'Transmission',
    'FUEL': 'Fuel type',
    'COMB (L/100 km)': 'Combined (L/100 km)',
    'EMISSIONS': 'CO2 emissions (g/km)'
})
print(data.head())

   YEAR   MAKE    MODEL Vehicle class  Engine size (L)  Cylinders  \
0  2000  ACURA    1.6EL       COMPACT              1.6          4   
1  2000  ACURA    1.6EL       COMPACT              1.6          4   
2  2000  ACURA    3.2TL      MID-SIZE              3.2          6   
3  2000  ACURA    3.5RL      MID-SIZE              3.5          6   
4  2000  ACURA  INTEGRA    SUBCOMPACT              1.8          4   

  Transmission Fuel type  FUEL CONSUMPTION  HWY (L/100 km)  \
0           A4         X               9.2             6.7   
1           M5         X               8.5             6.5   
2          AS5         Z              12.2             7.4   
3           A4         Z              13.4             9.2   
4           A4         X              10.0             7.0   

   Combined (L/100 km)  COMB (mpg)  CO2 emissions (g/km)  
0                  8.1          35                   186  
1                  7.6          37                   175  
2                 10.0          28 

In [30]:
data['Vehicle class'].value_counts()  # Extract first character

Vehicle class
SUV                         2640
COMPACT                     2636
MID-SIZE                    2300
PICKUP TRUCK - STANDARD     1689
SUBCOMPACT                  1559
FULL-SIZE                   1086
TWO-SEATER                   999
SUV: Small                   929
SUV - SMALL                  827
MINICOMPACT                  783
STATION WAGON - SMALL        737
Mid-size                     660
SUV: Standard                608
Pickup truck: Standard       515
SUV - STANDARD               514
Compact                      491
Subcompact                   451
Full-size                    417
PICKUP TRUCK - SMALL         403
MINIVAN                      366
STATION WAGON - MID-SIZE     343
VAN - CARGO                  332
Two-seater                   313
VAN - PASSENGER              287
Minicompact                  211
Station wagon: Small         140
Pickup truck: Small          108
Special purpose vehicle       62
SPECIAL PURPOSE VEHICLE       52
Station wagon: Mid-size      

In [31]:
def main():
    """Main function to run the ML workflow."""
    global lr_pipeline, rf_pipeline, gbr_pipeline  # Make models accessible outside function
    
    print("=" * 70)
    print("TRAINING MODELS ON: Fuel_Consumption_2000-2022.csv")
    print("=" * 70)

    numeric_features = ['Engine size (L)', 'Cylinders', 'Combined (L/100 km)']
    categorical_features = ['Fuel type', 'Vehicle class', 'Transmission']
    features = numeric_features + categorical_features
    target = 'CO2 emissions (g/km)'
    
    print(f"\nDataset shape: {data.shape}")
    print(f"Features: {features}")
    print(f"Target: {target}")

    X = data[features]
    y = data[target]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"\nData split: {len(X_train)} training samples, {len(X_test)} testing samples.")
    print("-" * 70)

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', 'passthrough', numeric_features),
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
        ]
    )

    # ===== LINEAR REGRESSION =====
    print("\n[1/3] Training Linear Regression model...")
    lr_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', LinearRegression())
    ])
    lr_pipeline.fit(X_train, y_train)
    lr_preds = lr_pipeline.predict(X_test)
    lr_mae = mean_absolute_error(y_test, lr_preds)
    lr_r2 = r2_score(y_test, lr_preds)
    print(f"      MAE: {lr_mae:.2f} g/km | R2: {lr_r2:.4f}")

    # ===== RANDOM FOREST =====
    print("\n[2/3] Training Random Forest model...")
    rf_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
    ])
    rf_pipeline.fit(X_train, y_train)
    rf_preds = rf_pipeline.predict(X_test)
    rf_mae = mean_absolute_error(y_test, rf_preds)
    rf_r2 = r2_score(y_test, rf_preds)
    print(f"      MAE: {rf_mae:.2f} g/km | R2: {rf_r2:.4f}")

    # ===== GRADIENT BOOSTING =====
    print("\n[3/3] Training Gradient Boosting model...")
    gbr_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=5, random_state=42))
    ])
    gbr_pipeline.fit(X_train, y_train)
    gbr_preds = gbr_pipeline.predict(X_test)
    gbr_mae = mean_absolute_error(y_test, gbr_preds)
    gbr_r2 = r2_score(y_test, gbr_preds)
    print(f"      MAE: {gbr_mae:.2f} g/km | R2: {gbr_r2:.4f}")

    # ===== SAVE ALL MODELS =====
    print("\n" + "=" * 70)
    print("SAVING MODELS...")
    print("=" * 70)
    
    joblib.dump(lr_pipeline, 'lr_model.joblib')
    print("lr_model.joblib  (Linear Regression)")
    
    joblib.dump(rf_pipeline, 'rf_model.joblib')
    print("rf_model.joblib  (Random Forest)")
    
    joblib.dump(gbr_pipeline, 'gbr_model.joblib')
    print("gbr_model.joblib (Gradient Boosting)")
    
    # Default model
    joblib.dump(rf_pipeline, 'co2_model.joblib')  # Using RF as default (best performance)
    print("co2_model.joblib (Default: Random Forest)")
    
    print("\n" + "=" * 70)
    print("MODEL PERFORMANCE COMPARISON")
    print("=" * 70)
    print(f"{'Model':<25} {'MAE (g/km)':<15} {'R2 Score':<15}")
    print("-" * 70)
    print(f"{'Linear Regression':<25} {lr_mae:<15.2f} {lr_r2:<15.4f}")
    print(f"{'Random Forest':<25} {rf_mae:<15.2f} {rf_r2:<15.4f} <- BEST")
    print(f"{'Gradient Boosting':<25} {gbr_mae:<15.2f} {gbr_r2:<15.4f}")
    print("=" * 70)
    
    # Test on first training sample
    print("\nTESTING ON FIRST TRAINING SAMPLE:")
    print("-" * 70)
    test_sample = X_train.iloc[[0]]
    actual_val = y_train.iloc[0]
    
    lr_test = lr_pipeline.predict(test_sample)[0]
    rf_test = rf_pipeline.predict(test_sample)[0]
    gbr_test = gbr_pipeline.predict(test_sample)[0]
    
    print(f"Actual CO2:           {actual_val:.2f} g/km")
    print(f"LR Prediction:        {lr_test:.2f} g/km (Error: {abs(actual_val - lr_test):.2f})")
    print(f"RF Prediction:        {rf_test:.2f} g/km (Error: {abs(actual_val - rf_test):.2f})")
    print(f"GBR Prediction:       {gbr_test:.2f} g/km (Error: {abs(actual_val - gbr_test):.2f})")
    print("=" * 70)
    
    print("\nALL MODELS SAVED! Ready to use in Flask app.")
    print("   To switch models in app.py, change MODEL_CHOICE variable.")
    print("=" * 70)
    
    return lr_pipeline, rf_pipeline, gbr_pipeline

In [32]:
if __name__ == "__main__":
    main()

TRAINING MODELS ON: Fuel_Consumption_2000-2022.csv

Dataset shape: (22556, 13)
Features: ['Engine size (L)', 'Cylinders', 'Combined (L/100 km)', 'Fuel type', 'Vehicle class', 'Transmission']
Target: CO2 emissions (g/km)

Data split: 18044 training samples, 4512 testing samples.
----------------------------------------------------------------------

[1/3] Training Linear Regression model...
      MAE: 2.21 g/km | R2: 0.9937

[2/3] Training Random Forest model...
      MAE: 0.76 g/km | R2: 0.9991

[3/3] Training Gradient Boosting model...
      MAE: 0.76 g/km | R2: 0.9991

[3/3] Training Gradient Boosting model...
      MAE: 1.04 g/km | R2: 0.9991

SAVING MODELS...
lr_model.joblib  (Linear Regression)
rf_model.joblib  (Random Forest)
gbr_model.joblib (Gradient Boosting)
co2_model.joblib (Default: Random Forest)

MODEL PERFORMANCE COMPARISON
Model                     MAE (g/km)      R2 Score       
----------------------------------------------------------------------
Linear Regression   

In [33]:
# VERIFY: Test all saved models are different
print("=" * 70)
print("VERIFYING SAVED MODELS ARE DIFFERENT")
print("=" * 70)

# Load all models
lr_loaded = joblib.load('lr_model.joblib')
rf_loaded = joblib.load('rf_model.joblib')
gbr_loaded = joblib.load('gbr_model.joblib')

# Test input
test_input = pd.DataFrame({
    'Engine size (L)': [3.5],
    'Cylinders': [6],
    'Combined (L/100 km)': [10.5],
    'Fuel type': ['X'],
    'Vehicle class': ['COMPACT'],
    'Transmission': ['A6']
})

# Get predictions
lr_result = lr_loaded.predict(test_input)[0]
rf_result = rf_loaded.predict(test_input)[0]
gbr_result = gbr_loaded.predict(test_input)[0]

print("\nTest Input:")
print(test_input)

print("\n" + "=" * 70)
print("PREDICTIONS FROM SAVED MODELS:")
print("=" * 70)
print(f"Linear Regression:   {lr_result:.2f} g/km")
print(f"Random Forest:       {rf_result:.2f} g/km")
print(f"Gradient Boosting:   {gbr_result:.2f} g/km")
print("-" * 70)
print(f"Max Difference:      {max(lr_result, rf_result, gbr_result) - min(lr_result, rf_result, gbr_result):.2f} g/km")
print("=" * 70)

if abs(lr_result - rf_result) > 1 or abs(lr_result - gbr_result) > 1:
    print("Models are DIFFERENT - Ready for deployment!")
else:
    print("Models are very similar - consider different hyperparameters")
    
print("=" * 70)

VERIFYING SAVED MODELS ARE DIFFERENT

Test Input:
   Engine size (L)  Cylinders  Combined (L/100 km) Fuel type Vehicle class  \
0              3.5          6                 10.5         X       COMPACT   

  Transmission  
0           A6  

PREDICTIONS FROM SAVED MODELS:
Linear Regression:   242.01 g/km
Random Forest:       242.00 g/km
Gradient Boosting:   243.12 g/km
----------------------------------------------------------------------
Max Difference:      1.12 g/km
Models are DIFFERENT - Ready for deployment!


## TEST DATA SET AND SMAPE SCORE FOR ACCURACY

In [34]:
# Generate synthetic test dataset
import numpy as np

np.random.seed(42)

# Sample roughly the same size as training data (22,556 rows)
n_samples = 22000

# Create synthetic test data based on realistic ranges from the original dataset
test_data = pd.DataFrame({
    'Engine size (L)': np.random.uniform(1.0, 8.0, n_samples),
    'Cylinders': np.random.choice([3, 4, 5, 6, 8, 10, 12, 16], n_samples),
    'Combined (L/100 km)': np.random.uniform(4.0, 25.0, n_samples),
    'Fuel type': np.random.choice(['X', 'Z', 'D', 'E'], n_samples),
    'Vehicle class': np.random.choice(['COMPACT', 'MID-SIZE', 'SUV - SMALL', 'SUV - STANDARD', 
                                       'SUBCOMPACT', 'FULL-SIZE', 'STATION WAGON - SMALL',
                                       'PICKUP TRUCK - STANDARD', 'VAN - PASSENGER'], n_samples),
    'Transmission': np.random.choice(['A4', 'A5', 'A6', 'A7', 'A8', 'A9', 'A10', 
                                      'M5', 'M6', 'M7', 'AS4', 'AS5', 'AS6', 
                                      'AS7', 'AS8', 'AS9', 'AS10', 'AM5', 'AM6', 'AM7'], n_samples)
})

# Generate realistic CO2 emissions based on a formula (approximation)
# CO2 roughly correlates with fuel consumption and engine size
test_data['CO2 emissions (g/km)'] = (
    test_data['Combined (L/100 km)'] * 23.0 + 
    test_data['Engine size (L)'] * 5.0 + 
    np.random.normal(0, 10, n_samples)  # Add some noise
).clip(100, 500)  # Realistic CO2 range

print(f"Synthetic test dataset created: {len(test_data)} rows")
print(test_data.head())
print(f"\nCO2 emissions range: {test_data['CO2 emissions (g/km)'].min():.1f} - {test_data['CO2 emissions (g/km)'].max():.1f}")

Synthetic test dataset created: 22000 rows
   Engine size (L)  Cylinders  Combined (L/100 km) Fuel type  \
0         3.621781         10            14.689337         X   
1         7.655000         12            10.429947         E   
2         6.123958          8            23.565491         D   
3         5.190609          4             8.794049         D   
4         2.092130          8            12.818289         D   

           Vehicle class Transmission  CO2 emissions (g/km)  
0  STATION WAGON - SMALL           M5            352.189469  
1             SUBCOMPACT           A4            290.003867  
2  STATION WAGON - SMALL           M6            500.000000  
3                COMPACT          A10            234.972986  
4        VAN - PASSENGER         AS10            304.281708  

CO2 emissions range: 100.0 - 500.0


In [35]:
# Define SMAPE (Symmetric Mean Absolute Percentage Error) function
def calculate_smape(y_true, y_pred):
    """
    Calculate SMAPE: Symmetric Mean Absolute Percentage Error
    Range: 0% (perfect) to 200% (worst)
    Formula: 100 * mean(|y_true - y_pred| / ((|y_true| + |y_pred|) / 2))
    """
    numerator = np.abs(y_true - y_pred)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    # Avoid division by zero
    smape = np.mean(np.where(denominator == 0, 0, numerator / denominator)) * 100
    return smape

print("SMAPE function defined")

SMAPE function defined


In [36]:
# Evaluate all models on synthetic test data with SMAPE

# Prepare test features and target
X_test_synthetic = test_data[['Engine size (L)', 'Cylinders', 'Combined (L/100 km)', 
                                'Fuel type', 'Vehicle class', 'Transmission']]
y_test_synthetic = test_data['CO2 emissions (g/km)']

print("=" * 60)
print("EVALUATING ALL MODELS ON SYNTHETIC TEST DATA")
print("=" * 60)

# Note: We need to run main() first to train the models
# The models (lr_pipeline, rf_pipeline, gbr_pipeline) should exist in the namespace

try:
    # Linear Regression Predictions
    lr_test_preds = lr_pipeline.predict(X_test_synthetic)
    lr_test_mae = mean_absolute_error(y_test_synthetic, lr_test_preds)
    lr_test_r2 = r2_score(y_test_synthetic, lr_test_preds)
    lr_test_smape = calculate_smape(y_test_synthetic.values, lr_test_preds)
    
    print("\nLINEAR REGRESSION - Test Data Performance")
    print(f"   MAE:    {lr_test_mae:.2f} g/km")
    print(f"   R2:     {lr_test_r2:.4f}")
    print(f"   SMAPE:  {lr_test_smape:.2f}%")
    print("-" * 60)
    
    # Random Forest Predictions
    rf_test_preds = rf_pipeline.predict(X_test_synthetic)
    rf_test_mae = mean_absolute_error(y_test_synthetic, rf_test_preds)
    rf_test_r2 = r2_score(y_test_synthetic, rf_test_preds)
    rf_test_smape = calculate_smape(y_test_synthetic.values, rf_test_preds)
    
    print("\nRANDOM FOREST - Test Data Performance")
    print(f"   MAE:    {rf_test_mae:.2f} g/km")
    print(f"   R2:     {rf_test_r2:.4f}")
    print(f"   SMAPE:  {rf_test_smape:.2f}%")
    print("-" * 60)
    
    # Gradient Boosting Predictions
    gbr_test_preds = gbr_pipeline.predict(X_test_synthetic)
    gbr_test_mae = mean_absolute_error(y_test_synthetic, gbr_test_preds)
    gbr_test_r2 = r2_score(y_test_synthetic, gbr_test_preds)
    gbr_test_smape = calculate_smape(y_test_synthetic.values, gbr_test_preds)
    
    print("\nGRADIENT BOOSTING - Test Data Performance")
    print(f"   MAE:    {gbr_test_mae:.2f} g/km")
    print(f"   R2:     {gbr_test_r2:.4f}")
    print(f"   SMAPE:  {gbr_test_smape:.2f}%")
    print("-" * 60)
    
    # Summary comparison
    print("\nSMAPE COMPARISON (Lower is Better)")
    print(f"   Linear Regression:   {lr_test_smape:.2f}%")
    print(f"   Random Forest:       {rf_test_smape:.2f}%")
    print(f"   Gradient Boosting:   {gbr_test_smape:.2f}%")
    print("=" * 60)
    
except NameError as e:
    print(f"Error: {e}")
    print("Please run main() first to train all models before evaluating on test data.")

EVALUATING ALL MODELS ON SYNTHETIC TEST DATA

LINEAR REGRESSION - Test Data Performance
   MAE:    48.80 g/km
   R2:     0.7014
   SMAPE:  21.54%
------------------------------------------------------------



RANDOM FOREST - Test Data Performance
   MAE:    42.95 g/km
   R2:     0.7598
   SMAPE:  13.86%
------------------------------------------------------------

GRADIENT BOOSTING - Test Data Performance
   MAE:    41.45 g/km
   R2:     0.7713
   SMAPE:  13.71%
------------------------------------------------------------

SMAPE COMPARISON (Lower is Better)
   Linear Regression:   21.54%
   Random Forest:       13.86%
   Gradient Boosting:   13.71%

GRADIENT BOOSTING - Test Data Performance
   MAE:    41.45 g/km
   R2:     0.7713
   SMAPE:  13.71%
------------------------------------------------------------

SMAPE COMPARISON (Lower is Better)
   Linear Regression:   21.54%
   Random Forest:       13.86%
   Gradient Boosting:   13.71%
